# Doc

> Defines a doc object interface to a Google Sheet. 

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| default_exp doc

In [ ]:
#| hide
#| export
from gspread import oauth
import pandas as pd
from nbdev import show_doc
from fastcore.test import *
from fastcore.basics import *
from fastcore.foundation import *
from numbers_parser import Document

In [ ]:
#| export
class Doc:
    def __init__(
            self,
            sheet_name:str, # name of the google sheet to access
            creds:str='credentials.json' # filename containing google auth credentials
        ):
        self._gc = oauth(credentials_filename=creds)
        self.type = 'Google Sheet'
        self.doc = self._gc.open(sheet_name)

### Extracting a dataframe from a worksheet

Each tab in a google sheet is known as a worksheet. We can extract a single worksheet using the `get_dataframe` function.

The result is a dataframe that can contain header information based on the first for of the worksheet (or not).

In [ ]:
#| export
@patch
def get_dataframe(
        self:Doc, 
        worksheet:str,     # name of the worksheet to return
        headers:bool=True  # return headers?
    )->pd.DataFrame:       # dataframe containing worksheet data 
    """ returns a dataframe from a worksheet """
    wks = self.doc.worksheet(worksheet)
    data = wks.get_all_values()
    if headers:
        headers = data.pop(0)
    else:
        headers = []
    df = pd.DataFrame(data, columns=headers)
    return df

In [ ]:
#| export
from typing import List
@patch
def get_worksheet_names(
    self:Doc,
    )->List[str]:
    """ returns a list of worksheet names """
    return [wks.title for wks in self.doc.worksheets()]


## Initialize a new document from a google sheet

We need to specify a sheet name and the location of a credentials file:


In [ ]:
# sheet_name = "Caylor CV"
# credentials_filename = 'credentials.json'
# doc = Doc(sheet_name,creds=credentials_filename)

In [ ]:
#df = doc.get_dataframe('Summary')

In [ ]:
# test_eq(doc.get_worksheet_names(), 
    # ['Summary', 'Publications', 'Other Pubs', 'Conference Abstracts', 'Lectures', 
    # 'Citations', 'Graduate Advising', 'Undergrad', 'Postdoc', 'Reviews', 'Additional Professional Activity', 
    # 'Service', 'Special Appointments', 'Visitors', 'Grants', 'Courses', 'MESM Projects', 'Biography', 'Awards', 'Data']
    # )

In [ ]:
# test_eq(list(df.columns),['Current Year', '2023'])

## Numbers Documents

A subclass of `Doc` that allows the use of a Numbers spreadsheet in biobib and other CV/website generation.

In [ ]:
# | hide
# | export
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [ ]:
# | export
NUMBERS_PATH = os.getenv("NUMBERS_PATH")

In [ ]:
# | export
class NumbersDoc(Doc):
    def __init__(
        self,
        sheet_name: str,  # name of the numbers document to access
        path: str = NUMBERS_PATH,  # path to the numbers document
        default_table: str = "Table" # default name of the table to access (defaults to "Table")
    ):
        full_path = os.path.join(path, sheet_name)
        self.doc = Document(full_path)
        self.type = 'Numbers Document'
        self.default_table = default_table
        self._sheets = {}
        [self._sheets.update({sheet.name: sheet}) for sheet in self.doc.sheets]

    def get_worksheet_names(self):
        return list(self._sheets.keys())

    def get_dataframe(self, worksheet, 
                      data_table_name=None # Data table name is the name of the table to access inside the Numbers document. Only change if the table name is different from the default_table!
                      ):
        if data_table_name is None:
            data_table = self.default_table
        data = self._sheets[worksheet].tables[data_table].rows(values_only=True)
        return pd.DataFrame(data[1:], columns=data[0])

In [ ]:
sheet_name = "CV.numbers"
num_doc = NumbersDoc(sheet_name)

In [ ]:
test_eq(num_doc.get_worksheet_names(),
        ['Summary', 'Publications', 'Citations', 'Other Pubs', 'Conference Abstracts', 'Media', 'Lectures', 
          'Graduate PhD', 'Postdoc', 'Graduate MA/MS', 'Reviews', 'Additional Professional Activities', 'Service', 
          'Special Appointments', 'Undergrad', 'Visitors', 'Grants', 'Teaching', 'MESM Projects', 
          'Biography', 'Awards', 'Data'
]
        )

In [ ]:
df = num_doc.get_dataframe("Publications")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()